# Notebook 2: Snowflake Setup — Data Upload, Feature Store & Online Config

**Objective:** Upload the synthetic healthcare data to Snowflake, set up the Feature Store with a
Patient entity, create a Feature View backed by a Dynamic Table, and enable the Online Feature Store
for real-time serving.

**Prerequisites:** Run Notebook 01 first to generate artifacts in `./artifacts/`

**Snowflake objects created:**
- Tables: `HEALTHCARE_ML.RAW_DATA.PATIENTS`, `ADMISSIONS`, `CLINICAL_MEASUREMENTS`
- Entity: `PATIENT` (join key: `PATIENT_ID`)
- Feature View: `PATIENT_CLINICAL_FEATURES` (backed by Dynamic Table, 5-min refresh)
- Online Feature Store enabled with 1-minute target lag

In [ ]:
import os
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
ARTIFACTS_DIR = os.path.join(PROJECT_DIR, 'artifacts')

# Load metadata from Notebook 01
with open(os.path.join(ARTIFACTS_DIR, 'model_metadata.json')) as f:
    metadata = json.load(f)
print(f"Loaded metadata: {metadata['training_samples']} training samples, AUC={metadata['model_metrics']['roc_auc']}")

## 1. Connect to Snowflake

We use Snowpark to establish a session. The connection uses the DEMO connection profile.

In [ ]:
from snowflake.snowpark import Session

connection_params = {
    'connection_name': 'DEMO'
}

session = Session.builder.configs(connection_params).create()
session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql('USE WAREHOUSE HEALTHCARE_ML_WH').collect()
session.sql('USE DATABASE HEALTHCARE_ML').collect()

print(f"Connected: {session.get_current_account()}")
print(f"Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Database: {session.get_current_database()}")

## 2. Upload Raw Data to Snowflake

Load the CSV artifacts from Notebook 01 and write them as Snowflake tables.
This simulates a hospital's EHR data landing in the data warehouse.

In [ ]:
# Load CSVs
patients_df = pd.read_csv(os.path.join(ARTIFACTS_DIR, 'patients.csv'))
admissions_df = pd.read_csv(os.path.join(ARTIFACTS_DIR, 'admissions.csv'))
clinical_df = pd.read_csv(os.path.join(ARTIFACTS_DIR, 'clinical.csv'))

print(f"Patients: {patients_df.shape}")
print(f"Admissions: {admissions_df.shape}")
print(f"Clinical: {clinical_df.shape}")

In [ ]:
# Write to Snowflake tables in RAW_DATA schema
session.sql('USE SCHEMA RAW_DATA').collect()

sp_patients = session.create_dataframe(patients_df)
sp_patients.write.mode('overwrite').save_as_table('PATIENTS')
print(f"PATIENTS table: {session.table('PATIENTS').count()} rows")

sp_admissions = session.create_dataframe(admissions_df)
sp_admissions.write.mode('overwrite').save_as_table('ADMISSIONS')
print(f"ADMISSIONS table: {session.table('ADMISSIONS').count()} rows")

sp_clinical = session.create_dataframe(clinical_df)
sp_clinical.write.mode('overwrite').save_as_table('CLINICAL_MEASUREMENTS')
print(f"CLINICAL_MEASUREMENTS table: {session.table('CLINICAL_MEASUREMENTS').count()} rows")

In [ ]:
# Also upload the pre-engineered training and test data for model registration later
training_df = pd.read_csv(os.path.join(ARTIFACTS_DIR, 'training_data.csv'))
test_data_df = pd.read_csv(os.path.join(ARTIFACTS_DIR, 'test_data.csv'))

sp_training = session.create_dataframe(training_df)
sp_training.write.mode('overwrite').save_as_table('TRAINING_DATA')
print(f"TRAINING_DATA table: {session.table('TRAINING_DATA').count()} rows")

sp_test = session.create_dataframe(test_data_df)
sp_test.write.mode('overwrite').save_as_table('TEST_DATA')
print(f"TEST_DATA table: {session.table('TEST_DATA').count()} rows")

## 3. Verify Uploaded Data

In [ ]:
# Quick verification
print("=== PATIENTS ===")
session.table('PATIENTS').show(3)

print("\n=== ADMISSIONS ===")
session.table('ADMISSIONS').show(3)

print("\n=== CLINICAL_MEASUREMENTS ===")
session.table('CLINICAL_MEASUREMENTS').show(3)

## 4. Set Up the Feature Store

The Snowflake Feature Store provides:
- **Entities**: Logical groupings (e.g., Patient) with join keys
- **Feature Views**: Collections of features backed by Dynamic Tables for auto-refresh
- **Online Store**: Low-latency feature serving for real-time inference

We create a `PATIENT` entity and a `PATIENT_CLINICAL_FEATURES` feature view that joins
patients, admissions, and clinical data into a single feature set.

In [ ]:
from snowflake.ml.feature_store import (
    FeatureStore, Entity, FeatureView, CreationMode
)

# Initialize Feature Store (uses the FEATURE_STORE schema)
fs = FeatureStore(
    session=session,
    database='HEALTHCARE_ML',
    name='FEATURE_STORE',
    default_warehouse='HEALTHCARE_ML_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

print("Feature Store initialized.")

In [ ]:
# Register the Patient entity
patient_entity = Entity(
    name='PATIENT',
    join_keys=['PATIENT_ID'],
    desc='Hospital patient entity for readmission prediction'
)
fs.register_entity(patient_entity)

print("Registered entities:")
print(fs.list_entities().to_pandas())

In [ ]:
# Build the feature query — join patients + admissions + clinical into a single feature set
# This SQL computes the same features we engineered locally in Notebook 01
feature_query = """
SELECT
    a.PATIENT_ID,
    a.ADMISSION_ID,
    TO_TIMESTAMP(a.DISCHARGE_DATE) AS EVENT_TIMESTAMP,
    -- Patient demographics
    p.AGE,
    CASE WHEN p.GENDER = 'M' THEN 1 ELSE 0 END AS GENDER_ENC,
    CASE p.INSURANCE_TYPE
        WHEN 'MEDICAID' THEN 0 WHEN 'MEDICARE' THEN 1
        WHEN 'PRIVATE' THEN 2 WHEN 'SELF_PAY' THEN 3
    END AS INSURANCE_ENC,
    CASE WHEN p.HAS_PCP THEN 1 ELSE 0 END AS HAS_PCP_FLAG,
    -- Admission features
    a.LENGTH_OF_STAY,
    a.NUM_PROCEDURES,
    a.NUM_DIAGNOSES,
    CASE a.PRIMARY_DIAGNOSIS
        WHEN 'HEART_FAILURE' THEN 3 WHEN 'SEPSIS' THEN 3
        WHEN 'COPD' THEN 2 WHEN 'RENAL_FAILURE' THEN 2
        WHEN 'ACUTE_MI' THEN 2 WHEN 'DIABETES_COMPLICATIONS' THEN 2 WHEN 'STROKE' THEN 2
        ELSE 1
    END AS DIAGNOSIS_RISK_SCORE,
    CASE a.DISCHARGE_DISPOSITION
        WHEN 'HOME' THEN 0 WHEN 'HOME_HEALTH' THEN 1 WHEN 'REHAB' THEN 1
        WHEN 'SNF' THEN 2 WHEN 'AMA' THEN 3
    END AS DISPOSITION_RISK_SCORE,
    a.ED_ADMISSION,
    -- Clinical measurements at discharge
    c.HEART_RATE,
    c.SYSTOLIC_BP,
    c.DIASTOLIC_BP,
    c.TEMPERATURE,
    c.RESPIRATORY_RATE,
    c.O2_SATURATION,
    c.BLOOD_GLUCOSE,
    c.CREATININE,
    c.HEMOGLOBIN,
    c.WBC_COUNT,
    c.SODIUM,
    c.POTASSIUM,
    c.BNP,
    -- Clinical abnormality flags
    CASE WHEN c.HEART_RATE > 100 OR c.HEART_RATE < 60 THEN 1 ELSE 0 END AS ABNORMAL_HR,
    CASE WHEN c.SYSTOLIC_BP > 160 OR c.SYSTOLIC_BP < 90 THEN 1 ELSE 0 END AS ABNORMAL_BP,
    CASE WHEN c.O2_SATURATION < 93 THEN 1 ELSE 0 END AS LOW_O2,
    CASE WHEN c.CREATININE > 1.5 THEN 1 ELSE 0 END AS HIGH_CREATININE,
    CASE WHEN c.HEMOGLOBIN < 10 THEN 1 ELSE 0 END AS LOW_HEMOGLOBIN,
    CASE WHEN c.BNP > 500 THEN 1 ELSE 0 END AS HIGH_BNP,
    CASE WHEN c.BLOOD_GLUCOSE > 200 OR c.BLOOD_GLUCOSE < 70 THEN 1 ELSE 0 END AS ABNORMAL_GLUCOSE,
    -- Historical features (computed via window functions)
    COALESCE(COUNT(*) OVER (
        PARTITION BY a.PATIENT_ID
        ORDER BY a.ADMIT_DATE
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ), 0) AS PRIOR_ADMISSIONS_6M,
    COALESCE(SUM(a.READMITTED_30D) OVER (
        PARTITION BY a.PATIENT_ID
        ORDER BY a.ADMIT_DATE
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ), 0) AS PRIOR_READMISSIONS,
    COALESCE(AVG(a.LENGTH_OF_STAY) OVER (
        PARTITION BY a.PATIENT_ID
        ORDER BY a.ADMIT_DATE
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ), 0) AS AVG_PRIOR_LOS,
    -- Target (for training dataset generation)
    a.READMITTED_30D
FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS a
JOIN HEALTHCARE_ML.RAW_DATA.PATIENTS p ON a.PATIENT_ID = p.PATIENT_ID
JOIN HEALTHCARE_ML.RAW_DATA.CLINICAL_MEASUREMENTS c
    ON a.ADMISSION_ID = c.ADMISSION_ID AND a.PATIENT_ID = c.PATIENT_ID
"""

feature_df = session.sql(feature_query)
print(f"Feature query columns: {len(feature_df.columns)}")
feature_df.show(3)

In [ ]:
# Create the Feature View — backed by a Dynamic Table that auto-refreshes
patient_features_fv = FeatureView(
    name='PATIENT_CLINICAL_FEATURES',
    entities=[patient_entity],
    feature_df=feature_df,
    timestamp_col='EVENT_TIMESTAMP',
    refresh_freq='5 minutes',
    desc='Patient demographics, admission details, clinical vitals, and historical features for readmission prediction'
)

# Register the feature view (creates the underlying Dynamic Table)
patient_features_fv = fs.register_feature_view(
    feature_view=patient_features_fv,
    version='V1',
    overwrite=True
)

print("Feature View registered successfully.")
print(f"Feature View: {patient_features_fv.name}")
print(f"Version: {patient_features_fv.version}")

In [ ]:
# List feature views to confirm
print("Registered Feature Views:")
print(fs.list_feature_views().to_pandas()[['NAME', 'VERSION', 'DESC']].to_string())

## 5. Enable Online Feature Store

The Online Feature Store provides low-latency feature lookups for real-time inference.
It maintains a synchronized copy of the feature data optimized for point lookups by entity key.

In [ ]:
# Enable Online Feature Store on the feature view
# This creates an online-optimized replica with near-real-time sync
from snowflake.ml.feature_store.feature_view import OnlineConfig

fs.update_feature_view(
    name='PATIENT_CLINICAL_FEATURES',
    version='V1',
    online_config=OnlineConfig(enable=True, target_lag='1 minute')
)

print("Online Feature Store enabled for PATIENT_CLINICAL_FEATURES (target lag: 1 minute)")

## 6. Generate a Training Dataset (for lineage tracking)

Even though we trained locally, generating a dataset through the Feature Store creates
lineage from raw data -> features -> training dataset -> model.

In [ ]:
# Create a spine DataFrame (entity keys + timestamps for point-in-time join)
spine_df = session.sql("""
    SELECT PATIENT_ID, TO_TIMESTAMP(DISCHARGE_DATE) AS EVENT_TIMESTAMP
    FROM HEALTHCARE_ML.RAW_DATA.ADMISSIONS
    ORDER BY DISCHARGE_DATE
""")

print(f"Spine rows: {spine_df.count()}")
spine_df.show(3)

In [ ]:
# Retrieve the registered feature view
fv = fs.get_feature_view('PATIENT_CLINICAL_FEATURES', 'V1')

# Generate the training dataset with full lineage
dataset = fs.generate_dataset(
    name='READMISSION_TRAINING_V1',
    spine_df=spine_df,
    features=[fv],
    spine_timestamp_col='EVENT_TIMESTAMP',
    desc='Training dataset for 30-day readmission prediction model'
)

# Verify the dataset
ds_df = dataset.read.to_snowpark_dataframe()
print(f"Generated dataset rows: {ds_df.count()}")
print(f"Generated dataset columns: {len(ds_df.columns)}")
ds_df.show(3)

## 7. Verification Summary

In [ ]:
print("="*60)
print("SNOWFLAKE SETUP VERIFICATION")
print("="*60)

# Check tables
tables = session.sql("SHOW TABLES IN SCHEMA HEALTHCARE_ML.RAW_DATA").collect()
print(f"\nRAW_DATA tables: {len(tables)}")
for t in tables:
    print(f"  - {t['name']}: {t['rows']} rows")

# Check feature store
print(f"\nFeature Store entities: {len(fs.list_entities().to_pandas())}")
print(f"Feature Store views: {len(fs.list_feature_views().to_pandas())}")

# Check dynamic table status
try:
    dt_status = session.sql("""
        SELECT NAME, SCHEDULING_STATE, DATA_TIMESTAMP
        FROM TABLE(HEALTHCARE_ML.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY())
        WHERE NAME LIKE '%PATIENT_CLINICAL%'
        ORDER BY DATA_TIMESTAMP DESC
        LIMIT 1
    """).collect()
    if dt_status:
        print(f"\nDynamic Table status: {dt_status[0]['SCHEDULING_STATE']}")
        print(f"Last refresh: {dt_status[0]['DATA_TIMESTAMP']}")
except Exception as e:
    print(f"\nDynamic Table refresh history not yet available (normal for new DTs): {e}")

print("\nSetup complete. Proceed to Notebook 03 for model registration.")

session.close()